<a href="https://colab.research.google.com/github/gayathrikakumani24-droid/Artificial-Neural-Networks-ANN-/blob/main/ANN_CyberForce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module I: Practice Exercise - 'Cyber-Force AI' (Advanced Spec)

### **Context: Network Intrusion Detection**
You are the Lead Security Engineer for 'Cyber-Force'. You have traffic logs from 500 devices.
Your sensors measure:
1.  **Packet Size (Bytes)**
2.  **Request Latency (ms)**
3.  **Encryption Entropy (0-10)**

You have **two separate goals**:
* **Goal A (Severity):** Predict the **Threat Severity Score (0-100)**. (Regression)
* **Goal B (Attack Type):** Classify the traffic: **0=Benign, 1=DDoS, 2=Phishing**. (Multi-Class Classification)

---
**INSTRUCTIONS:**
This exercise uses **Early Stopping** and **L2 Regularization** to handle overfitting.
The architectures are **Deeper (3+ layers)** and **Wider (128+ neurons)**.

In [ ]:
# CELL 1: DATA GENERATION (Run this first)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(999)

# 500 Traffic Samples
N = 500
# Features: Size (64-1500), Latency (10-500), Entropy (0-10)
X = torch.rand(N, 3) * torch.tensor([1436, 490, 10]) + torch.tensor([64, 10, 0])

# Target A: Severity Score (Regression)
# Severity increases with Entropy and Latency
y_sev = (X[:, 1] * 0.1) + (X[:, 2] * 5) + torch.randn(N) * 5
y_sev = y_sev.view(-1, 1).float()

# Target B: Attack Type (Multi-Class 0, 1, 2)
# Logic: High Latency -> DDoS (1), High Entropy -> Phishing (2), Else Benign (0)
y_type = torch.zeros(N).long()
y_type[X[:, 1] > 300] = 1 # DDoS
y_type[X[:, 2] > 8] = 2   # Phishing

print(f"Data Ready. X: {X.shape}, y_sev: {y_sev.shape}, y_type: {y_type.shape}")

Data Ready. X: torch.Size([500, 3]), y_sev: torch.Size([500, 1]), y_type: torch.Size([500])


## **Level 1: Severity Prediction (Deep Regression)**
**Your Task:** Predict Threat Severity.

**Blueprint (Architecture):**
1.  **Input:** 3 Features
2.  **Layer 1:** 64 Neurons, `ReLU`
3.  **Layer 2:** 32 Neurons, `ReLU`
4.  **Layer 3:** 16 Neurons, `ReLU`
5.  **Output:** 1 Neuron (Linear)

**Training Specs:**
* Loss: `MSELoss`
* Optimizer: `Adam` (lr=0.005)
* Epochs: 150

In [ ]:
# LEVEL 1: WRITE YOUR CODE HERE

# 1. Define 'SeverityModel' (3 Hidden Layers!)
class SeverityModel(nn.Module):
    # TODO: Linear(3,64) -> ReLU -> Linear(64,32) -> ReLU -> Linear(32,16) -> ReLU -> Linear(16,1)
   def __init__(self):
    super(SeverityModel,self).__init__()
    self.fc1=nn.Linear(3,64)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(64,32)
    self.relu=nn.ReLU()
    self.fc3=nn.Linear(32,16)
    self.relu=nn.ReLU()
    self.fc4=nn.Linear(16,1)
   def forward(self,x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    x=self.relu(x)
    x=self.fc3(x)
    x=self.relu(x)
    x=self.fc4(x)
    return x

model_sev = SeverityModel()

# 2. Optimizer & Loss
loss_fn=nn.MSELoss()
optimizer=optim.Adam(model_sev.parameters(),lr=0.005)
epochs=150
# 3. Training Loop
train_loss=0
for epoch in range(epochs):
  pred=model_sev(X)
  loss=loss_fn(pred,y_sev)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  train_loss+=loss.item()
# 4. Check Loss
  if epoch%20==0:
    print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 1153.7001953125
Epoch 20, Loss: 307.7303771972656
Epoch 40, Loss: 287.9227294921875
Epoch 60, Loss: 271.038330078125
Epoch 80, Loss: 244.80975341796875
Epoch 100, Loss: 197.98866271972656
Epoch 120, Loss: 148.2084503173828
Epoch 140, Loss: 61.73748779296875


In [ ]:
# TEST LEVEL 1
try:
    # Check Depth
    layers = [m for m in model_sev.modules() if isinstance(m, nn.Linear)]
    assert len(layers) == 4, f"Blueprint requires 4 Linear layers. Found {len(layers)}"
    assert layers[0].out_features == 64
    assert layers[1].out_features == 32

    print("✅ Level 1 Passed: Deep Regression Architecture Correct.")
except Exception as e:
    print(f"❌ Level 1 Fail: {e}")

✅ Level 1 Passed: Deep Regression Architecture Correct.


## **Level 2: Attack Classification (Overfitting & Early Stopping)**
**Your Task:** Predict Attack Type (0, 1, 2).

**Step 2.1: Pipeline**
* Split: **50 Train / 450 Test** (Tiny training set to force overfitting!).
* `DataLoader` (Batch=10).

**Step 2.2: The 'Overkill' Model**
* `Linear(3, 128) -> ReLU`
* `Linear(128, 128) -> ReLU`
* `Linear(128, 3)`

**Step 2.3: Diagnosis**
* Train for 200 epochs. Observe that Test Loss goes UP while Train Loss goes DOWN.

**Step 2.4: The Fix (Early Stopping & L2 Regularization)**
* **Fix 1:** Add `weight_decay=0.01` to the Adam optimizer (L2 Regularization).
* **Fix 2:** Implement **Early Stopping** inside the loop:
    * If `test_loss` does not improve for 10 epochs (patience), `break` the loop.
    * *Hint: Keep track of `best_loss`.*

In [ ]:
# LEVEL 2: WRITE YOUR CODE HERE

# 1. Pipeline (50 Train / 450 Test)
X_train,X_test=X[:50],X[50:]
y_train,y_test=y_sev[:50],y_sev[50:]
train_ds=TensorDataset(X_train,y_train)
val_ds=TensorDataset(X_test,y_test)

train_loader=DataLoader(train_ds,batch_size=10,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=10)
# 2. Define 'model_overkill'
class model_Overkill(nn.Module):
  def __init__(self):
    super(model_Overkill,self).__init__()
    self.fc1=nn.Linear(3,128)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(128,128)
    self.relu=nn.ReLU()
    self.fc3=nn.Linear(128,1)
  def forward(self,x):
    x=self.fc1(x)
    x=self.relu(x)
    x=self.fc2(x)
    x=self.relu(x)
    x=self.fc3(x)
    return x
# 3. Optimizer WITH L2 Regularization (weight_decay=0.01)
model_overkill=model_Overkill()
# opt_l2 = optim.Adam(model_Overkill.parameters(),lr=0.01)
opt_l2 = optim.Adam(model_overkill.parameters(),lr=0.01,weight_decay=0.01)
loss_fn=nn.MSELoss()
# 4. Training Loop WITH Early Stopping
best_loss = float('inf')
patience = 10
trigger_times = 0
train_loss=0
val_loss=0

for epoch in range(200):
  model_overkill.train()
  for x_batch,y_batch in train_loader:
    pred=model_overkill(x_batch)
    loss=loss_fn(pred,y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_loss+=loss.item()
  model_overkill.eval()
  with torch.no_grad():
    pred=model_overkill(x_batch)
    loss=loss_fn(pred,y_batch)
    val_loss+=loss.item()
  if epoch%20==0:
    print(f"Epoch: {epoch}, Training Loss: {train_loss}, Test Loss: {val_loss}")
  if val_loss<best_loss:
    trigger_times=0
    best_loss=val_loss
  else:
    trigger_times+=1
    if(trigger_times>=patience):
      print("Early Stopping")
      break
# for epoch in range(200):
#     Train...
#     Calc Test Loss...
#     Early Stopping Logic:
#         if test_loss < best_loss:
#             best_loss = test_loss
#             trigger_times = 0
#         else:
#             trigger_times += 1
#             if trigger_times >= patience:
#                 print("Early Stopping!")
#                 break

Epoch: 0, Training Loss: 68076.84375, Test Loss: 14602.978515625
Early Stopping


In [ ]:
# TEST LEVEL 2
try:
    # Check L2
    assert opt_l2.defaults['weight_decay'] == 0.01, "Optimizer must have weight_decay=0.01"

    # Check Width
    l1 = list(model_overkill.modules())[1]
    if isinstance(l1, nn.Sequential): l1 = l1[0]
    assert l1.out_features == 128, "Hidden layer must be 128 wide"

    print("✅ Level 2 Passed: Early Stopping & L2 Logic Valid.")
except Exception as e:
    print(f"❌ Level 2 Fail: {e}")

✅ Level 2 Passed: Early Stopping & L2 Logic Valid.


## **Level 3: The Tournament (Activations)**

**Part 3.1: Manual CrossEntropy (Concept)**
Remember: `CrossEntropyLoss` combines `LogSoftmax` and `NLLLoss`.

**Part 3.2: The Tournament**
Compare 3 configs on **Attack Classification**:

1.  **"Baseline"**: `ReLU` + `Adam` (Standard)
2.  **"Dying ReLU Fix"**: `LeakyReLU` + `SGD`
3.  **"Smooth"**: `ELU` + `Adam`
    * *Note: ELU (Exponential Linear Unit) can often converge faster than ReLU.*

In [ ]:
# LEVEL 3: TOURNAMENT
experiments = [
    {"name": "Baseline", "act": nn.ReLU(), "opt": torch.optim.Adam(model_overkill.parameters(),lr=0.01)}, # ReLU, Adam
    {"name": "Leaky",    "act": nn.LeakyReLU(), "opt": torch.optim.SGD(model_overkill.parameters(),lr=0.01,momentum=0.9)}, # LeakyReLU, SGD
    {"name": "Smooth",   "act": nn.ELU(), "opt": torch.optim.Adam(model_overkill.parameters(),lr=0.01)}  # ELU, Adam
]

print(f"{'Name':<10} | {'Test Acc':<10}")
print("-"*25)

# Loop...

Name       | Test Acc  
-------------------------


In [ ]:
# TEST LEVEL 3
try:
    assert isinstance(experiments[1]['act'], nn.LeakyReLU), "Exp 2 Act must be LeakyReLU"
    assert isinstance(experiments[2]['act'], nn.ELU), "Exp 3 Act must be ELU"

    print("✅ Level 3 Passed: Advanced Activations Configured.")
except Exception as e:
    print(f"❌ Level 3 Fail: {e}")

✅ Level 3 Passed: Advanced Activations Configured.


## **Level 4: The Mechanic (2-Input Gradient Descent)**
**Your Task:** Train a model manually (No `optim` library).

**Goal:** Learn $y = 0.5x_1 - 2.0x_2 + 10$

1.  **Data:** $X$ has shape `(N, 2)`.
2.  **Model:** `nn.Linear(2, 1)`.
3.  **Update Rule:** Manual Gradient Descent.
4.  **Important:** Use `with torch.no_grad():`.

In [ ]:
# LEVEL 4: THE PURGE
import torch.optim as optim
import gc
for var in list(locals().keys()):
    if 'opt' in var or 'optimizer' in var:
        del locals()[var]
gc.collect()
print("Optimizers deleted. You are on your own.")

In [ ]:
# LEVEL 4: WRITE YOUR CODE HERE

# 1. Setup Data (2 Inputs)
X_vec = torch.tensor([[1.0, 1.0], [2.0, 4.0], [5.0, 1.0]])
# Target: 0.5*x1 - 2.0*x2 + 10
y_vec = (0.5 * X_vec[:, 0] - 2.0 * X_vec[:, 1] + 10).view(-1, 1)



# 2. Define Model
model_vec = nn.Linear(2,1)
loss_fn=nn.MSELoss()
lr=0.05
epochs=500
# 3. Manual Loop
    # Forward, Loss, Backward
for epoch in range(epochs):
  pred=model_vec(X_vec)
  loss=loss_fn(pred,y_vec)
  loss.backward()
    # Update
  with torch.no_grad():
    model_vec.weight -=lr*model_vec.weight.grad
    model_vec.bias -= lr* model_vec.bias.grad
    # Zero grad
    model_vec.weight.grad.zero_()
    model_vec.bias.grad.zero_()

    if epoch % 50==0:
      print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Epoch 0 | Loss: 38.3425
Epoch 50 | Loss: 3.7302
Epoch 100 | Loss: 1.0043
Epoch 150 | Loss: 0.2704
Epoch 200 | Loss: 0.0728
Epoch 250 | Loss: 0.0196
Epoch 300 | Loss: 0.0053
Epoch 350 | Loss: 0.0014
Epoch 400 | Loss: 0.0004
Epoch 450 | Loss: 0.0001


In [ ]:
# TEST LEVEL 4
try:
    w = model_vec.weight.data[0]
    b = model_vec.bias.item()

    print(f"Final Weights: {w}")
    print(f"Final Bias: {b}")

    assert abs(w[0] - 0.5) < 0.2, f"w1 should be ~0.5"
    assert abs(w[1] + 2.0) < 0.2, f"w2 should be ~ -2.0"
    assert abs(b - 10.0) < 0.2, f"Bias should be ~10.0"

    print("✅ Level 4 Passed: 2-Input Gradient Descent successful!")
except Exception as e:
    print(f"❌ Level 4 Fail: {e}")

Final Weights: tensor([ 0.5025, -1.9972])
Final Bias: 9.985908508300781
✅ Level 4 Passed: 2-Input Gradient Descent successful!
